# 🌐 MYGRATE - AI Multi-Agent Conversational Sandbox Demo

Welcome to the **Mygrate** interactive graduation thesis prototype! 

This Jupyter Notebook serves as the Proof-of-Concept (PoC) demonstrating how Mygrate's Multi-Agent architecture coordinates code indexing, version auditing, and compatibility solving in a **Semi-Automated (Human-in-the-Loop)** workflow.

### 🗣️ Conversational-First Design & Human-in-the-Loop:
Unlike fully automated pipelines, the **Supervisor Agent (Central Orchestrator)** communicates directly with the user. It remains in conversational chat mode by default. You can chat with the Supervisor, send commands, and approve plans—all from the **Single Interactive Chat Cell** at the bottom of the notebook!

---

## ⚙️ Step 1: Environment Setup & Core Imports

In [1]:
import os
import json
import sys
from dotenv import load_dotenv
from IPython.display import Image, display, Markdown, clear_output

# Ensure the src folder is in the python path
sys.path.append(os.path.abspath("."))

# Load env variables (GROQ_API_KEY, etc.)
load_dotenv()
print("[+] Setup completed. Mygrate core imports successfully loaded.")

[+] Setup completed. Mygrate core imports successfully loaded.


## 📂 Step 2: Project Selection & Configuration

Configure the target project directory and environmental constraints. In this demo, we use the `cantor` open-source repository as our testbed.

In [2]:
# Configure target project parameters
TARGET_PROJECT = "freshbrew_data/cantor"
TARGET_JAVA = "17"

print(f"[+] Target project configured: '{TARGET_PROJECT}'")
print(f"[+] Target JDK version: '{TARGET_JAVA}'")

[+] Target project configured: 'freshbrew_data/cantor'
[+] Target JDK version: '17'


## 🔄 Step 3: Initialize Workflow Thread

Initialize the Mygrate `GlobalState` context and config. The thread ID allows the memory saver to persist our conversation history.

In [3]:
from src.workflow import app
from src.models.state import GlobalState

initial_state = {
    "project_path": TARGET_PROJECT,
    "target_java_version": TARGET_JAVA,
    "source_framework": "spring-boot",
    "source_version": "2.2.0",
    "target_framework": "spring-boot",
    "target_version": "3.3.0",
    "messages": [],
    "completed_tasks_summary": [],
    "dependencies": [],
    "compatibility_matrix": {},
    "migration_tasks": [],
    "current_instruction": "",
    "last_subagent_result": "",
    "next_node": "supervisor"
}

config = {"configurable": {"thread_id": "mygrate_unified_chat_thread"}}

# Initialize graph with the initial state
app.invoke(initial_state, config=config)
print("[+] Global state and thread configuration initialized. Ready to chat!")

[+] Global state and thread configuration initialized. Ready to chat!


## 💬 Step 4: Interactive Agentic Chat Interface

Run the cell below repeatedly to chat with the Mygrate Supervisor. 
- **Greeting**: Type *"Chào bạn, tôi muốn nâng cấp dự án cũ"* (Supervisor will respond naturally).
- **Scan Command**: Type *"Hãy quét dự án cantor tại freshbrew_data/cantor"* (Supervisor will automatically execute all agent nodes under the hood, print the results, and render diagrams inline!).
- **Feedback/Approval**: Type *"Tôi đồng ý với phương án này"* to approve.

*Note: To continue the chat, just run this cell repeatedly.*

In [ ]:
from langchain_core.messages import HumanMessage

# 1. Prompt for user input
user_input = input("You: ")

if user_input.strip():
    # 2. Append user message to state
    feedback_message = HumanMessage(content=user_input)
    app.update_state(config, {"messages": [feedback_message]})

    print("\n[🤖 Mygrate is thinking...]")

    # 3. Resume graph execution (invokes Supervisor)
    app.invoke(None, config=config)

    # 4. Auto-run any routed sub-agents in sequence
    while True:
        state_val = app.get_state(config).values
        next_node = state_val.get("next_node", "end")
        if next_node == "end":
            break
        print(f"---> [Supervisor routing to sub-agent] Running: {next_node}...")
        app.invoke(None, config=config)

    # 5. Fetch final state values
    latest_state = app.get_state(config)
    last_messages = latest_state.values.get("messages", [])

    # 6. Clear cell output to keep chat clean
    clear_output(wait=True)

    # 7. Print Supervisor's response
    if last_messages:
        supervisor_response = last_messages[-1].content
        print(f"\n🤖 Mygrate Supervisor:\n{supervisor_response}\n")

    # 8. Check for generated compatibility matrix report, render inline
    intel_path = "migration_intelligence.json"
    last_result = latest_state.values.get("last_subagent_result", "")
    if last_result:
        import re, json
        # Try fenced JSON first, then fallback to any JSON-like object in the output
        json_match = re.search(r'```json\s*(\{.*?\})\s*```', last_result, re.DOTALL)
        candidate = None
        if json_match:
            candidate = json_match.group(1)
        else:
            # Try to find the first JSON object-looking substring
            brace_match = re.search(r'(\{[]*\})', last_result, re.DOTALL)
            if brace_match:
                candidate = brace_match.group(1)
        if candidate:
            try:
                parsed = json.loads(candidate)
                with open(intel_path, 'w', encoding='utf-8') as f:
                    json.dump(parsed, f, ensure_ascii=False, indent=2)
            except Exception as e:
                print("[!] Failed to parse JSON from Supervisor output:", e)

    if os.path.exists(intel_path):
        from src.tools.visualization_engine import generate_dashboard, generate_cross_matrix
        generate_dashboard(intel_path, "dependency_graph.png")
        generate_cross_matrix(intel_path, "cross_matrix.png")

        print("-" * 60)
        print("📈 DEPENDENCY RELATIONSHIP GRAPH:")
        display(Image("dependency_graph.png"))
        print("\n🌡️ CROSS-DEPENDENCY COMPATIBILITY HEATMAP:")
        display(Image("cross_matrix.png"))
        print("-" * 60)